In [1]:
import numpy as np
import pandas as pd


import torch 
import torch.nn as nn
import torch.optim as optim

from collections import Counter

In [2]:
# Step -1 Since out file is in tetx and its not in proper Format So we clean file and create new file 

input_file = "npi.txt"

clean_file = "Cleandata.txt"

normalized_file = "Normalized.txt"  #Same as clean but added Header and footer for Decoder Layer



In [4]:
with open (input_file,"r") as file_read:
    with open (clean_file,"w") as file_write:
        for line in file_read:
            parts = line.strip().split("\t")

            if len(parts) >=2:
                english = parts[0]
                nepali = parts[1]

                file_write.write(f"{english.lower()}\t{nepali}\n")
                

In [5]:
rough = {}
with open (clean_file,"r") as f:
   for l in f:
       parts = l.strip().split("\t")
       rough[parts[0]] = parts[1]


for key,value in rough.items():
    print(f"{key} ===> {value}")
    

    

who? ===> को?
hide. ===> लुक।
stay. ===> बस्नुहोस्।
hello! ===> नमस्ते!
smile. ===> मुस्कान।
attack! ===> आक्रमण!
go slow. ===> बिस्तारै जाउ।
i'm tom. ===> म टम हुँ।
find tom. ===> टमलाई खोज।
i am tom. ===> म टम हुँ।
i danced. ===> मैले नाचे।
who am i? ===> म को हु?
can i eat? ===> के म खान सक्छु ?
he is old. ===> उहाँ वृद्ध हुनुहुन्छ।
i am okay. ===> म ठिक छु।
i am sick. ===> म बिरामी छु।
i am sure. ===> म पक्का छु।
i am tall. ===> म अग्लो छु।
i am well. ===> म कुशल छु।
i'm crazy. ===> म पागल हुँ।
i'm sorry. ===> मलाई माफ गर्नुहोस्।
is it bad? ===> यो नराम्रो हो?
may i eat? ===> के म खान सक्छु ?
who is he? ===> ऊ को हो?
good night. ===> शुभ रात्री।
he is lazy. ===> ऊ अल्छी छ ।
he is nice. ===> उ राम्रो छ।
he is poor. ===> ऊ गरिब छ।
he is sick. ===> ऊ बिरामी छ।
how lovely! ===> कति प्यारो!
i am a boy. ===> म केटा हुँ।
i am a man. ===> म एक मानिस हुँ।
i am happy. ===> म खुशी छु।
i am human. ===> म मान्छे हुँ।
i am short. ===> म होचो छु।
i am smart. ===> म स्मार्ट छु।
i eat fish. ===> म 

In [6]:
# Adding Header and Footer in Data for Decoder Architecture (<Start> and  <Stop>)

# def normalization(text):
with open (clean_file,"r") as file_read:
        with open(normalized_file,"w") as file_write:
             for line in file_read:
                 parts = line.strip().split("\t")

                 if len(parts) >=2:
                     english = parts[0]
                     nepali = f"<SOS> {parts[1]} <EOS>"

                     file_write.write(f"{english}\t{nepali}\n")

                 
                 
    

In [7]:
# testing 
# with open (normalized_file ,"r") as f:
#     test = f.readline()
#     print(test)


rough = {}
with open (normalized_file,"r") as f:
   for l in f:
       parts = l.strip().split("\t")
       rough[parts[0]] = parts[1]


for key,value in rough.items():
    print(f"{key} ===> {value}")
    


who? ===> <SOS> को? <EOS>
hide. ===> <SOS> लुक। <EOS>
stay. ===> <SOS> बस्नुहोस्। <EOS>
hello! ===> <SOS> नमस्ते! <EOS>
smile. ===> <SOS> मुस्कान। <EOS>
attack! ===> <SOS> आक्रमण! <EOS>
go slow. ===> <SOS> बिस्तारै जाउ। <EOS>
i'm tom. ===> <SOS> म टम हुँ। <EOS>
find tom. ===> <SOS> टमलाई खोज। <EOS>
i am tom. ===> <SOS> म टम हुँ। <EOS>
i danced. ===> <SOS> मैले नाचे। <EOS>
who am i? ===> <SOS> म को हु? <EOS>
can i eat? ===> <SOS> के म खान सक्छु ? <EOS>
he is old. ===> <SOS> उहाँ वृद्ध हुनुहुन्छ। <EOS>
i am okay. ===> <SOS> म ठिक छु। <EOS>
i am sick. ===> <SOS> म बिरामी छु। <EOS>
i am sure. ===> <SOS> म पक्का छु। <EOS>
i am tall. ===> <SOS> म अग्लो छु। <EOS>
i am well. ===> <SOS> म कुशल छु। <EOS>
i'm crazy. ===> <SOS> म पागल हुँ। <EOS>
i'm sorry. ===> <SOS> मलाई माफ गर्नुहोस्। <EOS>
is it bad? ===> <SOS> यो नराम्रो हो? <EOS>
may i eat? ===> <SOS> के म खान सक्छु ? <EOS>
who is he? ===> <SOS> ऊ को हो? <EOS>
good night. ===> <SOS> शुभ रात्री। <EOS>
he is lazy. ===> <SOS> ऊ अल्छी छ । <EOS>
h

In [13]:

def build_vocab(sentence):
    vocab = {"<PAD>":0,"<SOS>":1,"EOS":2,"<UNK>":3}  # pad is added to help to control o which appear if batch doesnt have same seq lenght

    character = set("".join(sentence))  #get all unique character from data
    for char in character:
        if char not in vocab:
            vocab[char] = len(vocab)

    return vocab
               

In [14]:
# Encoding Data

# encoding mean converting numeric avlue for string like <sos> h <eos>   where sos -1  h-3 and eos-2

def encode(sentence, vocab):
    result = []

    # start token
    result.append(vocab["<SOS>"])

    # each character
    for char in sentence:
        result.append(vocab.get(char, vocab["<UNK>"]))

    # end token
    result.append(vocab["<EOS>"])

    return result
        